In [5]:
from pyomo.environ import *
import numpy as np
import os

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
INPUT_DIR = "Data"
OUTPUT_DIR = "Result"
K = 5
SOLVERS = ["scip", "cplex"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------------------------
# QUBO PREPROCESSING
# -------------------------------------------------
def preprocess_qubo(Q_raw):
    """
    Ensures the matrix is valid for QUBO solving.
    If symmetric: use as-is.
    If not symmetric: symmetrize using (Q + Q.T) / 2
    """
    assert Q_raw.ndim == 2, "Q must be a 2D matrix"
    assert Q_raw.shape[0] == Q_raw.shape[1], "Q matrix must be square"

    if np.allclose(Q_raw, Q_raw.T):
        return Q_raw, True
    else:
        Q_sym = 0.5 * (Q_raw + Q_raw.T)
        return Q_sym, False

# -------------------------------------------------
# MODEL BUILDER
# -------------------------------------------------
# Updated build_model to use the full symmetric matrix logic
def build_model(Q):
    n = Q.shape[0]
    model = ConcreteModel("QUBO_TopK")
    model.I = RangeSet(0, n - 1)
    model.x = Var(model.I, domain=Binary)

    # Use the full matrix sum since you already symmetrized it
    model.obj = Objective(
        expr=sum(Q[i, j] * model.x[i] * model.x[j] 
                 for i in range(n) for j in range(n)),
        sense=minimize
    )
    model.exclude = ConstraintList()
    return model


# -------------------------------------------------
# AUTOMATION
# -------------------------------------------------
matrix_files = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith(".txt"))
print(f"Found {len(matrix_files)} matrix files")

for matrix_file in matrix_files:

    qubo_path = os.path.join(INPUT_DIR, matrix_file)
    Q_raw = np.loadtxt(qubo_path)

    Q, is_symmetric = preprocess_qubo(Q_raw)
    n = Q.shape[0]

    output_path = os.path.join(OUTPUT_DIR, matrix_file)

    print("\n==============================================")
    print(f"Processing Matrix: {matrix_file}")
    print(f"Matrix size      : {n} x {n}")
    print(f"Symmetric        : {is_symmetric}")
    print("==============================================")

    with open(output_path, "w") as out:

        out.write(f"Matrix File : {matrix_file}\n")
        out.write(f"Matrix Size : {n} x {n}\n")
        out.write(f"Symmetric   : {is_symmetric}\n\n")

        if not is_symmetric:
            out.write("Matrix was NOT symmetric.\n")
            out.write("Converted to symmetric QUBO using (Q + Qᵀ)/2.\n\n")

        for solver_name in SOLVERS:

            solver = SolverFactory(solver_name)

            if not solver.available():
                out.write(f"{solver_name.upper()} not available.\n\n")
                continue

            print(f"\n--- Solving with {solver_name.upper()} ---")

            model = build_model(Q)
            solutions = []

            for k in range(K):

                result = solver.solve(model, tee=False)

                if result.solver.termination_condition != TerminationCondition.optimal:
                    print("No more optimal solutions.")
                    break


                bitstring = [int(round(value(model.x[i]))) for i in range(n)]
                obj_val = float(value(model.obj))

                solutions.append((obj_val, bitstring))

                print(f"Solution {k + 1}")
                print("Objective:", obj_val)
                print("Bitstring:", bitstring)

                # Exclude exact solution
                model.exclude.add(
                    sum(
                        (1 - model.x[i]) if bitstring[i] == 1 else model.x[i]
                        for i in range(n)
                    ) >= 1
                )

            if not solutions:
                continue

            best_obj, best_bits = min(solutions, key=lambda x: x[0])

            # -------------------------------
            # WRITE RESULTS
            # -------------------------------
            out.write("=" * 40 + "\n")
            out.write(f"Solver : {solver_name.upper()}\n")
            out.write("=" * 40 + "\n")
            out.write(f"Top-K Solutions : {len(solutions)}\n\n")

            for idx, (obj, bits) in enumerate(solutions, start=1):
                out.write(f"Solution {idx}\n")
                out.write(f"Objective : {obj}\n")
                out.write(f"Bitstring : {bits}\n\n")

            out.write("Best Solution\n")
            out.write(f"Objective : {best_obj}\n")
            out.write(f"Bitstring : {best_bits}\n\n")

    print(f"Results saved to: {output_path}")

print("\n✅ All matrices solved using mathematically correct QUBO logic.")


Found 2 matrix files

Processing Matrix: matrix_1.txt
Matrix size      : 4 x 4
Symmetric        : True

--- Solving with SCIP ---
Solution 1
Objective: -11.0
Bitstring: [1, 0, 0, 1]
Solution 2
Objective: -9.0
Bitstring: [0, 1, 0, 1]
Solution 3
Objective: -8.0
Bitstring: [0, 0, 1, 0]
Solution 4
Objective: -7.0
Bitstring: [0, 1, 1, 0]
Solution 5
Objective: -6.0
Bitstring: [0, 0, 0, 1]

--- Solving with CPLEX ---
Solution 1
Objective: -11.0
Bitstring: [1, 0, 0, 1]
Solution 2
Objective: -9.0
Bitstring: [0, 1, 0, 1]
Solution 3
Objective: -8.0
Bitstring: [0, 0, 1, 0]
Solution 4
Objective: -7.0
Bitstring: [0, 1, 1, 0]
Solution 5
Objective: -6.0
Bitstring: [0, 0, 0, 1]
Results saved to: Result/matrix_1.txt

Processing Matrix: matrix_2.txt
Matrix size      : 4 x 4
Symmetric        : True

--- Solving with SCIP ---
Solution 1
Objective: -20.0
Bitstring: [1, 1, 1, 1]
Solution 2
Objective: -15.0
Bitstring: [1, 0, 1, 1]
Solution 3
Objective: -9.0
Bitstring: [1, 1, 1, 0]
Solution 4
Objective: -9.0
B